# 12b WILDTRACK: MVDeTr Tracking + Person ReID Pipeline

**Pipeline**
1. Load MVDeTr ground-plane detections from 12a (`test.txt`)
2. Parse grid coordinates into world coordinates in centimeters
3. Run Hungarian temporal tracking on the ground plane
4. Project tracked positions to 7 camera views and save per-camera tracklets
5. Extract person ReID embeddings from projected camera crops
6. Score simple cross-camera merge candidates with cosine similarity
7. Evaluate WILDTRACK ground-plane metrics (MODA, MODP, IDF1, precision, recall)
8. Optionally sweep tracking hyperparameters

**Inputs**
- 12a kernel output: `gumfreddy/12a-wildtrack-mvdetr-training`
- WILDTRACK dataset: `aryashah2k/large-scale-multicamera-detection-dataset`
- Project repo: cloned at runtime from `MRKDaGods/gp` (branch `feature/people-tracking`)
- ReID weights: `gumfreddy/mtmc-weights`

**Outputs**
- `/kaggle/working/12b_output/ground_plane_tracks.json`
- `/kaggle/working/12b_output/tracklets/`
- `/kaggle/working/12b_output/global_trajectories.json`
- `/kaggle/working/12b_output/reid_features.npz`
- `/kaggle/working/12b_output/reid_merge_candidates.json`
- `/kaggle/working/12b_output/evaluation_summary.json`
- `/kaggle/working/12b_output/tracking_sweep_best.json`
- `/kaggle/working/debug.log`

In [ ]:
import atexit
import pathlib
import sys

_LOG_PATH = pathlib.Path("/kaggle/working/debug.log")
_LOG = _LOG_PATH.open("w", buffering=1)

class _TeeWriter:
    def __init__(self, original, log_file):
        self.original = original
        self.log_file = log_file

    def write(self, value):
        self.original.write(value)
        try:
            self.log_file.write(value)
        except Exception:
            pass

    def flush(self):
        self.original.flush()
        try:
            self.log_file.flush()
        except Exception:
            pass

sys.stdout = _TeeWriter(sys.stdout, _LOG)
sys.stderr = _TeeWriter(sys.stderr, _LOG)

@atexit.register
def _shutdown_log():
    try:
        _LOG.close()
    except Exception:
        pass

print("[12b] debug logging initialized")
print(f"[12b] writing log to {_LOG_PATH}")

In [ ]:
import json
import re
import shutil
import subprocess
import sys

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

gpu_info = []
if shutil.which("nvidia-smi"):
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=gpu_name,compute_cap", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode == 0:
        for raw_line in result.stdout.splitlines():
            line = raw_line.strip()
            if not line:
                continue
            name, _, capability = line.partition(",")
            match = re.search(r"(\d+)\.(\d+)", capability)
            if match:
                sm = int(match.group(1)) * 10 + int(match.group(2))
                gpu_info.append({"name": name.strip(), "capability": capability.strip(), "sm": sm})

if any(entry["sm"] < 70 for entry in gpu_info):
    pip_install(
        "torch==2.4.1",
        "torchvision==0.19.1",
        "--index-url",
        "https://download.pytorch.org/whl/cu124",
    )

pip_install("timm==0.9.16", "loguru", "omegaconf", "motmetrics", "scikit-learn", "scipy")

import torch

print(
    json.dumps(
        {
            "python": sys.version.split()[0],
            "torch": torch.__version__,
            "cuda_available": torch.cuda.is_available(),
            "device_count": torch.cuda.device_count(),
            "gpu_info": gpu_info,
        },
        indent=2,
    )
)

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

WORK_DIR = Path("/kaggle/working")
PROJECT = WORK_DIR / "gp"
REPO_URL = "https://github.com/MRKDaGods/gp.git"
BRANCH = "feature/people-tracking"

if not PROJECT.exists():
    print(f"Cloning {REPO_URL} branch={BRANCH} ...")
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "--branch", BRANCH,
        REPO_URL, str(PROJECT)
    ])
    print("Clone complete")
else:
    print(f"Project already exists at {PROJECT}")

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print(f"Working directory: {os.getcwd()}")
print(f"Python path includes: {PROJECT}")

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path


def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


requirements_path = PROJECT / "requirements.txt"
assert requirements_path.exists(), requirements_path
pip_install("-r", str(requirements_path))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"])


def print_kaggle_input_listing():
    input_root = Path("/kaggle/input")
    print("=== /kaggle/input directory listing ===")
    if not input_root.exists():
        print("  /kaggle/input does not exist")
        return []

    discovered_dirs = []
    for item_name in sorted(os.listdir(input_root)):
        full = input_root / item_name
        if full.is_dir():
            discovered_dirs.append(full)
            try:
                children = sorted(os.listdir(full))
            except OSError as exc:
                print(f"  {item_name}/ (<error: {exc}>)")
                continue
            print(f"  {item_name}/ ({len(children)} items)")
            for child_name in children[:10]:
                child_path = full / child_name
                try:
                    size = child_path.stat().st_size if child_path.is_file() else "dir"
                except OSError as exc:
                    size = f"error: {exc}"
                print(f"    {child_name} ({size})")
        else:
            try:
                size = full.stat().st_size
            except OSError as exc:
                size = f"error: {exc}"
            print(f"  {item_name} ({size})")

    return discovered_dirs


def file_size(path):
    try:
        return path.stat().st_size
    except OSError:
        return -1


def dedupe_paths(paths):
    unique_paths = []
    seen = set()
    for path in paths:
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        unique_paths.append(path)
    return unique_paths


INPUT_DIRS = print_kaggle_input_listing()
likely_12a_dirs = [
    path
    for path in INPUT_DIRS
    if "12a" in path.name.lower() or "mvdetr" in path.name.lower()
]
print(f"[12b] 12a-like input directories: {[path.name for path in likely_12a_dirs]}")

search_roots = likely_12a_dirs or INPUT_DIRS
test_matches = []
csv_matches = []
json_matches = []
checkpoint_matches = []
deform_dirs = []

for input_dir in search_roots:
    local_tests = [path for path in input_dir.rglob("test.txt") if path.is_file()]
    local_csvs = [path for path in input_dir.rglob("ground_plane_detections.csv") if path.is_file()]
    local_jsons = [path for path in input_dir.rglob("ground_plane_detections.json") if path.is_file()]
    local_checkpoints = [path for path in input_dir.rglob("MultiviewDetector.pth") if path.is_file()]
    local_deform_dirs = [
        path for path in input_dir.rglob("*") if path.is_dir() and "aug_deform_trans" in path.name.lower()
    ]
    if local_tests:
        print(f"[12b] test.txt matches under {input_dir.name}: {[str(path) for path in local_tests[:10]]}")
    if local_csvs:
        print(f"[12b] ground_plane_detections.csv matches under {input_dir.name}: {[str(path) for path in local_csvs[:10]]}")
    if local_jsons:
        print(f"[12b] ground_plane_detections.json matches under {input_dir.name}: {[str(path) for path in local_jsons[:10]]}")
    if local_checkpoints:
        print(f"[12b] checkpoint matches under {input_dir.name}: {[str(path) for path in local_checkpoints[:10]]}")
    if local_deform_dirs:
        print(f"[12b] aug_deform_trans dirs under {input_dir.name}: {[str(path) for path in local_deform_dirs[:10]]}")
    test_matches.extend(local_tests)
    csv_matches.extend(local_csvs)
    json_matches.extend(local_jsons)
    checkpoint_matches.extend(local_checkpoints)
    deform_dirs.extend(local_deform_dirs)

if not test_matches and not csv_matches and not json_matches and not checkpoint_matches and not deform_dirs:
    print("[12b] falling back to recursive search across all of /kaggle/input")
    input_root = Path("/kaggle/input")
    if input_root.exists():
        test_matches = [path for path in input_root.rglob("test.txt") if path.is_file()]
        csv_matches = [path for path in input_root.rglob("ground_plane_detections.csv") if path.is_file()]
        json_matches = [path for path in input_root.rglob("ground_plane_detections.json") if path.is_file()]
        checkpoint_matches = [path for path in input_root.rglob("MultiviewDetector.pth") if path.is_file()]
        deform_dirs = [
            path for path in input_root.rglob("*") if path.is_dir() and "aug_deform_trans" in path.name.lower()
        ]
        if test_matches:
            print(f"[12b] global test.txt matches: {[str(path) for path in test_matches[:10]]}")
        if csv_matches:
            print(f"[12b] global ground_plane_detections.csv matches: {[str(path) for path in csv_matches[:10]]}")
        if json_matches:
            print(f"[12b] global ground_plane_detections.json matches: {[str(path) for path in json_matches[:10]]}")
        if checkpoint_matches:
            print(f"[12b] global checkpoint matches: {[str(path) for path in checkpoint_matches[:10]]}")
        if deform_dirs:
            print(f"[12b] global aug_deform_trans dirs: {[str(path) for path in deform_dirs[:10]]}")

test_matches = dedupe_paths(test_matches)
csv_matches = dedupe_paths(csv_matches)
json_matches = dedupe_paths(json_matches)
checkpoint_matches = dedupe_paths(checkpoint_matches)
deform_dirs = dedupe_paths(deform_dirs)

TEST_TXT = max(test_matches, key=lambda path: (file_size(path), str(path))) if test_matches else None
GROUND_PLANE_CSV = max(csv_matches, key=lambda path: (file_size(path), str(path))) if csv_matches else None
GROUND_PLANE_JSON = max(json_matches, key=lambda path: (file_size(path), str(path))) if json_matches else None
MVDETR_CHECKPOINT = max(checkpoint_matches, key=lambda path: (file_size(path), str(path))) if checkpoint_matches else None
DETECTIONS_PATH = GROUND_PLANE_CSV or GROUND_PLANE_JSON or TEST_TXT

if GROUND_PLANE_CSV is not None:
    MVDETR_OUTPUT_DIR = GROUND_PLANE_CSV.parent
elif GROUND_PLANE_JSON is not None:
    MVDETR_OUTPUT_DIR = GROUND_PLANE_JSON.parent
elif TEST_TXT is not None:
    MVDETR_OUTPUT_DIR = TEST_TXT.parent
elif MVDETR_CHECKPOINT is not None:
    MVDETR_OUTPUT_DIR = MVDETR_CHECKPOINT.parent
elif deform_dirs:
    MVDETR_OUTPUT_DIR = deform_dirs[0]
else:
    MVDETR_OUTPUT_DIR = None

print(f"Found MVDeTr output: {MVDETR_OUTPUT_DIR}")
print(f"[12b] selected detections path: {DETECTIONS_PATH}")
print(f"[12b] raw test.txt path: {TEST_TXT}")
print(f"[12b] csv detections path: {GROUND_PLANE_CSV}")
print(f"[12b] json detections path: {GROUND_PLANE_JSON}")
print(f"[12b] MVDeTr checkpoint: {MVDETR_CHECKPOINT}")
input_contents = sorted(os.listdir("/kaggle/input")) if Path("/kaggle/input").exists() else []
assert MVDETR_OUTPUT_DIR is not None, f"Cannot find 12a output in /kaggle/input. Contents: {input_contents}"
assert DETECTIONS_PATH is not None, (
    "Cannot find any 12a detection export (ground_plane_detections.csv/json or test.txt) "
    f"under {MVDETR_OUTPUT_DIR}. Contents: {input_contents}"
)
print(f"[12b] using 12a detections: {DETECTIONS_PATH}")

required_dirs = ["Image_subsets", "annotations_positions", "calibrations"]


def is_wildtrack_root(path: Path) -> bool:
    return path.exists() and all((path / name).exists() for name in required_dirs)


def find_wildtrack_root():
    search_root = Path("/kaggle/input")
    if not search_root.exists():
        return None
    queue = [(search_root, 0)]
    while queue:
        current, depth = queue.pop(0)
        if is_wildtrack_root(current):
            return current
        if depth >= 4:
            continue
        try:
            children = [child for child in current.iterdir() if child.is_dir()]
        except OSError:
            continue
        for child in children:
            queue.append((child, depth + 1))
    return None


WILDTRACK_ROOT = find_wildtrack_root()
DATA_ROOT = WILDTRACK_ROOT
assert WILDTRACK_ROOT is not None, f"Cannot find WILDTRACK dataset in /kaggle/input. Contents: {input_contents}"
print(f"[12b] wildtrack root: {WILDTRACK_ROOT}")

IMAGE_SUBSETS_DIR = WILDTRACK_ROOT / "Image_subsets"
ANNOTATIONS_DIR = WILDTRACK_ROOT / "annotations_positions"
CALIBRATIONS_DIR = WILDTRACK_ROOT / "calibrations"
assert IMAGE_SUBSETS_DIR.is_dir(), IMAGE_SUBSETS_DIR
assert ANNOTATIONS_DIR.is_dir(), ANNOTATIONS_DIR
assert CALIBRATIONS_DIR.is_dir(), CALIBRATIONS_DIR

REID_DIR_CANDIDATES = [
    Path("/kaggle/input/datasets/gumfreddy/mtmc-weights/reid"),
    Path("/kaggle/input/mtmc-weights/reid"),
]
REID_DIR = next((path for path in REID_DIR_CANDIDATES if path.is_dir()), REID_DIR_CANDIDATES[0])
REID_CANDIDATES = [
    REID_DIR / "person_transreid_vit_base_market1501.pth",
    Path("/kaggle/input/datasets/gumfreddy/mtmc-weights/person_transreid_vit_base_market1501.pth"),
    Path("/kaggle/input/datasets/gumfreddy/mtmc-weights/models/person_transreid_vit_base_market1501.pth"),
    Path("/kaggle/input/mtmc-weights/reid/person_transreid_vit_base_market1501.pth"),
    Path("/kaggle/input/mtmc-weights/person_transreid_vit_base_market1501.pth"),
    PROJECT / "models" / "reid" / "person_transreid_vit_base_market1501.pth",
    PROJECT / "models" / "person_transreid_vit_base_market1501.pth",
]
if REID_DIR.is_dir():
    reid_weight_files = sorted(path.name for path in REID_DIR.glob("*.pth"))
    print(f"[12b] dataset reid .pth files: {reid_weight_files}")
else:
    print(f"[12b] dataset reid directory missing: {REID_DIR}")
REID_WEIGHTS = next((path for path in REID_CANDIDATES if path.is_file()), None)
if REID_WEIGHTS is None:
    print("[12b] WARNING: person ReID weights not found; feature extraction will be skipped")
else:
    print(f"[12b] reid weights: {REID_WEIGHTS}")

OUTPUT_DIR = WORK_DIR / "12b_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_MATCH_DISTANCE_CM = 100.0
MAX_MISSED_FRAMES = 3
MIN_TRACK_LENGTH = 3

USE_KALMAN_TRACKER = True
KALMAN_MAX_AGE = 2
KALMAN_MIN_HITS = 2
KALMAN_DISTANCE_GATE = 20.0
KALMAN_MAX_EUCLIDEAN_CM = 200.0
KALMAN_Q_STD = 8.0
KALMAN_R_STD = 8.0
KALMAN_TRACKING_SWEEP = {
    "max_age": [2, 3, 4, 5, 6, 8],
    "distance_gate": [15.0, 18.0, 20.0, 22.0, 25.0],
    "min_hits": [1, 2, 3],
    "q_std": [5.0, 6.0, 8.0, 10.0, 12.0],
    "r_std": [5.0, 6.0, 8.0, 10.0, 12.0],
    "interpolation_max_gap": [1, 2, 3, 4, 5],
}
# Detection confidence threshold for ground-plane evaluation
DETECTION_CONF_SWEEP = [0.15, 0.20, 0.25, 0.30, 0.35]
DETECTION_CONF_THRESHOLD = 0.25
INTERPOLATE_TRACKS = True
TRACK_INTERPOLATION_MAX_GAP = 3

FPS = 2.0
IMAGE_SIZE = (1920, 1080)
MATCH_THRESHOLD_CM = 50.0
NMS_RADIUS_CM = 50.0
REID_INPUT_SIZE = (256, 128)
REID_BATCH_SIZE = 32
MAX_CROPS_PER_TRACKLET = 8
REID_MERGE_THRESHOLD_SWEEP = [0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
REID_MERGE_THRESHOLD = min(REID_MERGE_THRESHOLD_SWEEP)

runtime_config = {
    "detections_path": str(DETECTIONS_PATH),
    "test_txt": str(TEST_TXT) if TEST_TXT is not None else None,
    "ground_plane_csv": str(GROUND_PLANE_CSV) if GROUND_PLANE_CSV is not None else None,
    "ground_plane_json": str(GROUND_PLANE_JSON) if GROUND_PLANE_JSON is not None else None,
    "mvdetr_checkpoint": str(MVDETR_CHECKPOINT) if MVDETR_CHECKPOINT is not None else None,
    "wildtrack_root": str(WILDTRACK_ROOT),
    "reid_weights": str(REID_WEIGHTS) if REID_WEIGHTS is not None else None,
    "output_dir": str(OUTPUT_DIR),
    "use_kalman_tracker": USE_KALMAN_TRACKER,
    "naive_tracking": {
        "max_match_distance_cm": MAX_MATCH_DISTANCE_CM,
        "max_missed_frames": MAX_MISSED_FRAMES,
        "min_track_length": MIN_TRACK_LENGTH,
    },
    "kalman_tracking": {
        "max_age": KALMAN_MAX_AGE,
        "min_hits": KALMAN_MIN_HITS,
        "distance_gate": KALMAN_DISTANCE_GATE,
        "max_euclidean_cm": KALMAN_MAX_EUCLIDEAN_CM,
        "q_std": KALMAN_Q_STD,
        "r_std": KALMAN_R_STD,
    },
    "track_interpolation": {
        "enabled": INTERPOLATE_TRACKS,
        "max_gap": TRACK_INTERPOLATION_MAX_GAP,
    },
    "reid_merge_threshold_sweep": REID_MERGE_THRESHOLD_SWEEP,
    "fps": FPS,
    "image_size": IMAGE_SIZE,
}
print(json.dumps(runtime_config, indent=2))

In [ ]:
from src.stage_wildtrack_mvdetr.pipeline import load_mvdetr_ground_plane_detections

detections = load_mvdetr_ground_plane_detections(
    DETECTIONS_PATH,
    normalize_wildtrack_frames=True,
)
assert detections, "No detections loaded from test.txt"

frame_ids = sorted({det.frame_id for det in detections})
raw_frame_ids = sorted({det.raw_frame_id for det in detections if det.raw_frame_id is not None})
per_frame = len(detections) / len(frame_ids)

print(f"[12b] detections: {len(detections)}")
print(f"[12b] normalized frames: {len(frame_ids)} ({frame_ids[0]} -> {frame_ids[-1]})")
if raw_frame_ids:
    print(f"[12b] raw wildtrack frames: {raw_frame_ids[0]} -> {raw_frame_ids[-1]}")
print(f"[12b] avg detections per frame: {per_frame:.2f}")

In [ ]:
# 12b Kalman ground-plane tracker
from collections import defaultdict
from dataclasses import dataclass, field

import numpy as np
from scipy.optimize import linear_sum_assignment

from src.stage_wildtrack_mvdetr.pipeline import (
    GroundPlaneDetection,
    GroundPlaneTrack,
    track_ground_plane_detections as track_ground_plane_detections_naive,
)


@dataclass
class KalmanTrack:
    track_id: int
    state: np.ndarray
    covariance: np.ndarray
    detections: list[GroundPlaneDetection] = field(default_factory=list)
    hits: int = 1
    consecutive_misses: int = 0
    last_frame_id: int = -1

    def is_confirmed(self, min_hits: int) -> bool:
        return self.hits >= min_hits

    def predict(self, frame_id: int, transition_matrix: np.ndarray, process_noise: np.ndarray) -> None:
        self.state = transition_matrix @ self.state
        self.covariance = transition_matrix @ self.covariance @ transition_matrix.T + process_noise
        self.last_frame_id = frame_id

    def update(
        self,
        detection: GroundPlaneDetection,
        measurement_matrix: np.ndarray,
        measurement_noise: np.ndarray,
    ) -> None:
        measurement = np.array([detection.x_cm, detection.y_cm], dtype=np.float64)
        innovation = measurement - (measurement_matrix @ self.state)
        innovation_covariance = measurement_matrix @ self.covariance @ measurement_matrix.T + measurement_noise
        kalman_gain = self.covariance @ measurement_matrix.T @ np.linalg.pinv(innovation_covariance)
        self.state = self.state + kalman_gain @ innovation
        identity = np.eye(self.covariance.shape[0], dtype=np.float64)
        self.covariance = (identity - kalman_gain @ measurement_matrix) @ self.covariance
        self.detections.append(detection)
        self.hits += 1
        self.consecutive_misses = 0


class KalmanGroundPlaneTracker:
    def __init__(
        self,
        max_age: int = 2,
        min_hits: int = 2,
        distance_gate: float = 20.0,
        max_euclidean_cm: float = 200.0,
        q_std: float = 8.0,
        r_std: float = 8.0,
    ) -> None:
        self.max_age = max_age
        self.min_hits = min_hits
        self.distance_gate = distance_gate
        self.max_euclidean_cm = max_euclidean_cm
        self.q_std = q_std
        self.r_std = r_std
        self.measurement_matrix = np.array(
            [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]],
            dtype=np.float64,
        )
        self.measurement_noise = np.diag([r_std**2, r_std**2]).astype(np.float64)
        self.active_tracks: dict[int, KalmanTrack] = {}
        self.finished_tracks: list[KalmanTrack] = []
        self.next_track_id = 0

    def _transition_matrix(self, dt: float) -> np.ndarray:
        return np.array(
            [[1.0, 0.0, dt, 0.0], [0.0, 1.0, 0.0, dt], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]],
            dtype=np.float64,
        )

    def _process_noise(self, dt: float) -> np.ndarray:
        q_var = float(self.q_std) ** 2
        dt2 = dt * dt
        dt3 = dt2 * dt
        dt4 = dt2 * dt2
        return q_var * np.array(
            [
                [dt4 / 4.0, 0.0, dt3 / 2.0, 0.0],
                [0.0, dt4 / 4.0, 0.0, dt3 / 2.0],
                [dt3 / 2.0, 0.0, dt2, 0.0],
                [0.0, dt3 / 2.0, 0.0, dt2],
            ],
            dtype=np.float64,
        )

    def _initial_covariance(self) -> np.ndarray:
        position_var = float(self.r_std) ** 2
        velocity_std = max(75.0, float(self.q_std) * 6.0)
        velocity_var = velocity_std**2
        return np.diag([position_var, position_var, velocity_var, velocity_var]).astype(np.float64)

    def _group_by_frame(self, detections):
        detections_by_frame = defaultdict(list)
        for detection in detections:
            detections_by_frame[int(detection.frame_id)].append(detection)
        return detections_by_frame

    def _mahalanobis_distance(self, track: KalmanTrack, detection: GroundPlaneDetection) -> tuple[float, float]:
        measurement = np.array([detection.x_cm, detection.y_cm], dtype=np.float64)
        innovation = measurement - (self.measurement_matrix @ track.state)
        innovation_covariance = (
            self.measurement_matrix @ track.covariance @ self.measurement_matrix.T + self.measurement_noise
        )
        distance_squared = float(innovation.T @ np.linalg.pinv(innovation_covariance) @ innovation)
        euclidean_cm = float(np.linalg.norm(innovation))
        return distance_squared, euclidean_cm

    def _spawn_track(self, detection: GroundPlaneDetection) -> None:
        state = np.array([detection.x_cm, detection.y_cm, 0.0, 0.0], dtype=np.float64)
        self.active_tracks[self.next_track_id] = KalmanTrack(
            track_id=self.next_track_id,
            state=state,
            covariance=self._initial_covariance(),
            detections=[detection],
            hits=1,
            consecutive_misses=0,
            last_frame_id=int(detection.frame_id),
        )
        self.next_track_id += 1

    def track(self, detections) -> list[GroundPlaneTrack]:
        detections_by_frame = self._group_by_frame(detections)
        if not detections_by_frame:
            return []

        first_frame_id = min(detections_by_frame)
        last_frame_id = max(detections_by_frame)
        for frame_id in range(first_frame_id, last_frame_id + 1):
            frame_detections = detections_by_frame.get(frame_id, [])
            active_items = list(self.active_tracks.items())

            for _, track in active_items:
                dt = max(1.0, float(frame_id - track.last_frame_id))
                track.predict(frame_id, self._transition_matrix(dt), self._process_noise(dt))

            matched_track_ids = set()
            matched_detection_indices = set()
            if active_items and frame_detections:
                cost_matrix = np.full((len(active_items), len(frame_detections)), np.inf, dtype=np.float64)
                for row, (_, track) in enumerate(active_items):
                    for col, detection in enumerate(frame_detections):
                        distance_squared, euclidean_cm = self._mahalanobis_distance(track, detection)
                        if distance_squared <= self.distance_gate and euclidean_cm <= self.max_euclidean_cm:
                            cost_matrix[row, col] = distance_squared

                if np.isfinite(cost_matrix).any():
                    assignment_costs = np.where(np.isfinite(cost_matrix), cost_matrix, 1.0e9)
                    row_ind, col_ind = linear_sum_assignment(assignment_costs)
                    for row, col in zip(row_ind, col_ind):
                        if not np.isfinite(cost_matrix[row, col]):
                            continue
                        track_id, track = active_items[row]
                        detection = frame_detections[col]
                        track.update(detection, self.measurement_matrix, self.measurement_noise)
                        matched_track_ids.add(track_id)
                        matched_detection_indices.add(col)

            stale_track_ids = []
            for track_id, track in list(self.active_tracks.items()):
                if track_id in matched_track_ids:
                    continue
                track.consecutive_misses += 1
                if track.consecutive_misses > self.max_age:
                    stale_track_ids.append(track_id)

            for track_id in stale_track_ids:
                self.finished_tracks.append(self.active_tracks.pop(track_id))

            for detection_index, detection in enumerate(frame_detections):
                if detection_index in matched_detection_indices:
                    continue
                self._spawn_track(detection)

        self.finished_tracks.extend(self.active_tracks.values())
        confirmed_tracks = [track for track in self.finished_tracks if track.is_confirmed(self.min_hits)]
        confirmed_tracks.sort(key=lambda track: track.track_id)
        return [
            GroundPlaneTrack(track_id=track.track_id, detections=list(track.detections))
            for track in confirmed_tracks
        ]




def _estimate_local_velocity(detections, index):
    previous_detection = detections[index - 1] if index > 0 else None
    current_detection = detections[index]
    next_detection = detections[index + 1] if index + 1 < len(detections) else None

    if previous_detection is not None and next_detection is not None:
        dt = max(1, int(next_detection.frame_id - previous_detection.frame_id))
        return (
            float(next_detection.x_cm - previous_detection.x_cm) / dt,
            float(next_detection.y_cm - previous_detection.y_cm) / dt,
        )
    if previous_detection is not None:
        dt = max(1, int(current_detection.frame_id - previous_detection.frame_id))
        return (
            float(current_detection.x_cm - previous_detection.x_cm) / dt,
            float(current_detection.y_cm - previous_detection.y_cm) / dt,
        )
    if next_detection is not None:
        dt = max(1, int(next_detection.frame_id - current_detection.frame_id))
        return (
            float(next_detection.x_cm - current_detection.x_cm) / dt,
            float(next_detection.y_cm - current_detection.y_cm) / dt,
        )
    return 0.0, 0.0


def _interpolate_position(start_pos, end_pos, start_velocity, end_velocity, ratio, gap):
    t = float(ratio)
    h00 = 2.0 * t**3 - 3.0 * t**2 + 1.0
    h10 = t**3 - 2.0 * t**2 + t
    h01 = -2.0 * t**3 + 3.0 * t**2
    h11 = t**3 - t**2
    return (
        h00 * float(start_pos)
        + h10 * float(start_velocity) * gap
        + h01 * float(end_pos)
        + h11 * float(end_velocity) * gap
    )


def interpolate_tracks(tracks, max_gap=3):
    """Interpolate short gaps using local motion cues when neighboring detections exist."""
    interpolated_tracks = []
    for track in tracks:
        detections = sorted(track.detections, key=lambda det: det.frame_id)
        if len(detections) < 2:
            interpolated_tracks.append(GroundPlaneTrack(track_id=track.track_id, detections=list(detections)))
            continue

        new_detections = []
        for index, detection in enumerate(detections):
            new_detections.append(detection)
            if index >= len(detections) - 1:
                continue

            next_detection = detections[index + 1]
            gap = int(next_detection.frame_id - detection.frame_id)
            if 1 < gap <= max_gap + 1:
                raw_gap = None
                if detection.raw_frame_id is not None and next_detection.raw_frame_id is not None:
                    raw_gap = int(next_detection.raw_frame_id - detection.raw_frame_id)
                start_velocity_x, start_velocity_y = _estimate_local_velocity(detections, index)
                end_velocity_x, end_velocity_y = _estimate_local_velocity(detections, index + 1)
                for step in range(1, gap):
                    ratio = step / gap
                    interpolated_raw_frame = None
                    if raw_gap is not None:
                        interpolated_raw_frame = int(round(detection.raw_frame_id + raw_gap * ratio))
                    new_detections.append(
                        GroundPlaneDetection(
                            frame_id=int(detection.frame_id + step),
                            x_cm=float(
                                _interpolate_position(
                                    start_pos=detection.x_cm,
                                    end_pos=next_detection.x_cm,
                                    start_velocity=start_velocity_x,
                                    end_velocity=end_velocity_x,
                                    ratio=ratio,
                                    gap=gap,
                                )
                            ),
                            y_cm=float(
                                _interpolate_position(
                                    start_pos=detection.y_cm,
                                    end_pos=next_detection.y_cm,
                                    start_velocity=start_velocity_y,
                                    end_velocity=end_velocity_y,
                                    ratio=ratio,
                                    gap=gap,
                                )
                            ),
                            score=float(detection.score * (1.0 - ratio) + next_detection.score * ratio),
                            raw_frame_id=interpolated_raw_frame,
                        )
                    )

        new_detections.sort(key=lambda det: (det.frame_id, det.x_cm, det.y_cm))
        interpolated_tracks.append(GroundPlaneTrack(track_id=track.track_id, detections=new_detections))

    return interpolated_tracks


def apply_track_interpolation(tracks, max_gap):
    base_detection_count = sum(len(track.detections) for track in tracks)
    interpolated_tracks = interpolate_tracks(tracks, max_gap=max_gap)
    interpolated_detection_count = sum(len(track.detections) for track in interpolated_tracks)
    return interpolated_tracks, interpolated_detection_count - base_detection_count


def track_ground_plane_kalman(
    detections,
    max_age: int = 2,
    min_hits: int = 2,
    distance_gate: float = 20.0,
    max_euclidean_cm: float = 200.0,
    q_std: float = 8.0,
    r_std: float = 8.0,
):
    tracker = KalmanGroundPlaneTracker(
        max_age=max_age,
        min_hits=min_hits,
        distance_gate=distance_gate,
        max_euclidean_cm=max_euclidean_cm,
        q_std=q_std,
        r_std=r_std,
    )
    return tracker.track(detections)


def run_ground_plane_tracker(detections):
    if USE_KALMAN_TRACKER:
        active_params = {
            "tracker": "kalman",
            "max_age": int(KALMAN_MAX_AGE),
            "min_hits": int(KALMAN_MIN_HITS),
            "distance_gate": float(KALMAN_DISTANCE_GATE),
            "max_euclidean_cm": float(KALMAN_MAX_EUCLIDEAN_CM),
            "q_std": float(KALMAN_Q_STD),
            "r_std": float(KALMAN_R_STD),
        }
        tracks = track_ground_plane_kalman(
            detections=detections,
            max_age=KALMAN_MAX_AGE,
            min_hits=KALMAN_MIN_HITS,
            distance_gate=KALMAN_DISTANCE_GATE,
            max_euclidean_cm=KALMAN_MAX_EUCLIDEAN_CM,
            q_std=KALMAN_Q_STD,
            r_std=KALMAN_R_STD,
        )
        return tracks, active_params

    active_params = {
        "tracker": "naive",
        "max_match_distance_cm": float(MAX_MATCH_DISTANCE_CM),
        "max_missed_frames": int(MAX_MISSED_FRAMES),
        "min_track_length": int(MIN_TRACK_LENGTH),
    }
    tracks = track_ground_plane_detections_naive(
        detections=detections,
        max_match_distance_cm=MAX_MATCH_DISTANCE_CM,
        max_missed_frames=MAX_MISSED_FRAMES,
        min_track_length=MIN_TRACK_LENGTH,
    )
    return tracks, active_params

In [ ]:
from pathlib import Path

from src.core.io_utils import save_global_trajectories, save_tracklets_by_camera
from src.core.wildtrack_calibration import load_wildtrack_calibration
from src.stage_wildtrack_mvdetr.pipeline import (
    _save_ground_plane_csv,
    _save_ground_plane_tracks,
    _tracks_to_projected_tracklets,
)

calibrations = load_wildtrack_calibration(CALIBRATIONS_DIR)
assert calibrations, f"No calibrations loaded from {CALIBRATIONS_DIR}"
print(f"[12b] calibration cameras: {sorted(calibrations)}")


def persist_tracking_outputs(selected_tracks):
    _save_ground_plane_tracks(selected_tracks, OUTPUT_DIR / "ground_plane_tracks.json")
    _save_ground_plane_csv(selected_tracks, OUTPUT_DIR / "ground_plane_tracks.csv")
    projected_tracklets, projected_trajectories = _tracks_to_projected_tracklets(
        tracks=selected_tracks,
        calibrations=calibrations,
        fps=FPS,
        image_size=IMAGE_SIZE,
    )
    save_tracklets_by_camera(projected_tracklets, OUTPUT_DIR / "tracklets")
    save_global_trajectories(projected_trajectories, OUTPUT_DIR / "global_trajectories.json")
    return projected_tracklets, projected_trajectories


tracks, ACTIVE_TRACKING_PARAMS = run_ground_plane_tracker(detections)
assert tracks, "Tracking produced zero valid tracks"

if INTERPOLATE_TRACKS:
    tracks, interpolated_detections_added = apply_track_interpolation(
        tracks,
        max_gap=TRACK_INTERPOLATION_MAX_GAP,
    )
else:
    interpolated_detections_added = 0
ACTIVE_TRACKING_PARAMS = {
    **ACTIVE_TRACKING_PARAMS,
    "interpolation_enabled": bool(INTERPOLATE_TRACKS),
    "interpolation_max_gap": int(TRACK_INTERPOLATION_MAX_GAP),
    "interpolated_detections_added": int(interpolated_detections_added),
}
track_lengths = sorted(len(track.detections) for track in tracks)
median_length = track_lengths[len(track_lengths) // 2]
print(f"[12b] tracker: {ACTIVE_TRACKING_PARAMS['tracker']}")
print(f"[12b] active tracking params: {json.dumps(ACTIVE_TRACKING_PARAMS, indent=2)}")
print(f"[12b] tracks: {len(tracks)}")
print(f"[12b] interpolated detections added: {interpolated_detections_added}")
print(
    f"[12b] track lengths: min={track_lengths[0]}, max={track_lengths[-1]}, "
    f"mean={sum(track_lengths) / len(track_lengths):.2f}, median={median_length}"
)

tracklets_by_camera, trajectories = persist_tracking_outputs(tracks)

total_tracklets = sum(len(items) for items in tracklets_by_camera.values())
print(f"[12b] projected tracklets: {total_tracklets}")
print(f"[12b] global trajectories: {len(trajectories)}")
for camera_id, camera_tracklets in sorted(tracklets_by_camera.items()):
    print(f"  {camera_id}: {len(camera_tracklets)} tracklets")

In [ ]:
import json
from pathlib import Path

import cv2
import numpy as np
from sklearn.preprocessing import normalize

from src.stage2_features.reid_model import ReIDModel

def sample_tracklet_frames(tracklet, max_samples=8):
    if len(tracklet.frames) <= max_samples:
        return tracklet.frames
    indices = np.linspace(0, len(tracklet.frames) - 1, num=max_samples, dtype=int)
    return [tracklet.frames[index] for index in indices]

camera_dirs = {
    path.name: path
    for path in sorted(IMAGE_SUBSETS_DIR.iterdir())
    if path.is_dir() and path.name.startswith("C")
}
print(f"[12b] image cameras: {sorted(camera_dirs)}")

REID_FEATURES = {}
REID_FEATURE_STATS = {}
if REID_WEIGHTS is None:
    print("[12b] skipping ReID feature extraction")
else:
    reid_model = ReIDModel(
        model_name="transreid",
        weights_path=str(REID_WEIGHTS),
        embedding_dim=768,
        input_size=REID_INPUT_SIZE,
        device="cuda:0" if __import__("torch").cuda.is_available() else "cpu",
        half=True,
        flip_augment=True,
        num_cameras=7,
        vit_model="vit_base_patch16_224",
        clip_normalization=False,
    )

    for trajectory in trajectories:
        crops = []
        crop_camera_ids = []
        for tracklet in trajectory.tracklets:
            camera_dir = camera_dirs.get(tracklet.camera_id)
            if camera_dir is None:
                continue
            sampled_frames = sample_tracklet_frames(tracklet, MAX_CROPS_PER_TRACKLET)
            for frame in sampled_frames:
                wildtrack_frame = frame.frame_id * 5
                image_path = camera_dir / f"{wildtrack_frame:08d}.png"
                if not image_path.is_file():
                    continue
                image = cv2.imread(str(image_path))
                if image is None:
                    continue
                x1, y1, x2, y2 = [int(round(value)) for value in frame.bbox]
                pad_x = max(4, int((x2 - x1) * 0.08))
                pad_y = max(4, int((y2 - y1) * 0.08))
                x1 = max(0, x1 - pad_x)
                y1 = max(0, y1 - pad_y)
                x2 = min(image.shape[1], x2 + pad_x)
                y2 = min(image.shape[0], y2 + pad_y)
                if x2 <= x1 or y2 <= y1:
                    continue
                crop = image[y1:y2, x1:x2]
                if crop.size == 0 or crop.shape[0] < 16 or crop.shape[1] < 8:
                    continue
                crops.append(crop)
                crop_camera_ids.append(int(tracklet.camera_id[1:]) - 1)

        if not crops:
            continue

        per_camera_embeddings = []
        unique_camera_ids = sorted(set(crop_camera_ids))
        for cam_id in unique_camera_ids:
            camera_crops = [crop for crop, crop_cam_id in zip(crops, crop_camera_ids) if crop_cam_id == cam_id]
            if not camera_crops:
                continue
            embeddings = reid_model.extract_features(
                camera_crops,
                batch_size=REID_BATCH_SIZE,
                cam_id=cam_id,
            )
            if embeddings.size == 0:
                continue
            per_camera_embeddings.append(embeddings.mean(axis=0))

        if not per_camera_embeddings:
            continue

        feature = np.mean(np.stack(per_camera_embeddings, axis=0), axis=0)
        feature = normalize(feature.reshape(1, -1), norm="l2")[0].astype(np.float32)
        REID_FEATURES[trajectory.global_id] = feature
        REID_FEATURE_STATS[trajectory.global_id] = {
            "num_crops": len(crops),
            "num_tracklets": len(trajectory.tracklets),
            "num_cameras": len({tracklet.camera_id for tracklet in trajectory.tracklets}),
        }

    print(f"[12b] extracted ReID features: {len(REID_FEATURES)} / {len(trajectories)} trajectories")

    if REID_FEATURES:
        np.savez(
            OUTPUT_DIR / "reid_features.npz",
            **{str(global_id): feature for global_id, feature in REID_FEATURES.items()},
        )
        with (OUTPUT_DIR / "reid_feature_stats.json").open("w", encoding="utf-8") as handle:
            json.dump(REID_FEATURE_STATS, handle, indent=2)

In [ ]:
import json
import math

merge_candidates = []
if REID_FEATURES:
    trajectory_by_id = {trajectory.global_id: trajectory for trajectory in trajectories}
    feature_items = sorted(REID_FEATURES.items())
    for index, (global_id_a, feature_a) in enumerate(feature_items):
        global_id_a = int(global_id_a)
        traj_a = trajectory_by_id.get(global_id_a)
        if traj_a is None:
            continue
        cameras_a = {tracklet.camera_id for tracklet in traj_a.tracklets}
        for global_id_b, feature_b in feature_items[index + 1 :]:
            global_id_b = int(global_id_b)
            traj_b = trajectory_by_id.get(global_id_b)
            if traj_b is None:
                continue
            cameras_b = {tracklet.camera_id for tracklet in traj_b.tracklets}
            shared_cameras = cameras_a & cameras_b
            score = float(np.dot(feature_a, feature_b))
            if score < REID_MERGE_THRESHOLD:
                continue
            time_a = traj_a.time_span
            time_b = traj_b.time_span
            temporal_overlap = max(0.0, min(time_a[1], time_b[1]) - max(time_a[0], time_b[0]))
            merge_candidates.append(
                {
                    "global_id_a": global_id_a,
                    "global_id_b": global_id_b,
                    "cosine_similarity": round(score, 6),
                    "shared_cameras": sorted(shared_cameras),
                    "temporal_overlap_s": round(temporal_overlap, 3),
                    "camera_count_a": len(cameras_a),
                    "camera_count_b": len(cameras_b),
                }
            )

    merge_candidates.sort(key=lambda item: item["cosine_similarity"], reverse=True)
    with (OUTPUT_DIR / "reid_merge_candidates.json").open("w", encoding="utf-8") as handle:
        json.dump(merge_candidates, handle, indent=2)

print(f"[12b] merge candidates above threshold {REID_MERGE_THRESHOLD}: {len(merge_candidates)}")
for candidate in merge_candidates[:10]:
    print(candidate)

In [ ]:
# 12b Kalman tracking sweep
import json

from src.stage5_evaluation.ground_plane_eval import evaluate_wildtrack_ground_plane

pred_frame_min = min(frame_ids)
pred_frame_max = max(frame_ids)


def tracking_result_key(record):
    return (record["idf1"], record["moda"], -record["id_switches"], -record["num_tracks"])


def evaluate_tracking_config(use_kalman, params):
    interpolation_max_gap = int(params.get("interpolation_max_gap", TRACK_INTERPOLATION_MAX_GAP))
    detection_conf_threshold = float(params.get("detection_conf_threshold", DETECTION_CONF_THRESHOLD))
    if use_kalman:
        sweep_tracks = track_ground_plane_kalman(
            detections=detections,
            max_age=params["max_age"],
            min_hits=params["min_hits"],
            distance_gate=params["distance_gate"],
            max_euclidean_cm=KALMAN_MAX_EUCLIDEAN_CM,
            q_std=params["q_std"],
            r_std=params["r_std"],
        )
        tracking_params = {
            "tracker": "kalman",
            "max_age": int(params["max_age"]),
            "min_hits": int(params["min_hits"]),
            "distance_gate": float(params["distance_gate"]),
            "max_euclidean_cm": float(KALMAN_MAX_EUCLIDEAN_CM),
            "q_std": float(params["q_std"]),
            "r_std": float(params["r_std"]),
            "interpolation_enabled": True,
            "interpolation_max_gap": interpolation_max_gap,
            "detection_conf_threshold": detection_conf_threshold,
        }
    else:
        sweep_tracks = track_ground_plane_detections_naive(
            detections=detections,
            max_match_distance_cm=params["max_match_distance_cm"],
            max_missed_frames=params["max_missed_frames"],
            min_track_length=params["min_track_length"],
        )
        tracking_params = {
            "tracker": "naive",
            "max_match_distance_cm": float(params["max_match_distance_cm"]),
            "max_missed_frames": int(params["max_missed_frames"]),
            "min_track_length": int(params["min_track_length"]),
            "interpolation_enabled": True,
            "interpolation_max_gap": interpolation_max_gap,
            "detection_conf_threshold": detection_conf_threshold,
        }

    sweep_tracks, interpolation_added = apply_track_interpolation(
        sweep_tracks,
        max_gap=interpolation_max_gap,
    )
    sweep_tracklets_by_camera, sweep_trajectories = _tracks_to_projected_tracklets(
        tracks=sweep_tracks,
        calibrations=calibrations,
        fps=FPS,
        image_size=IMAGE_SIZE,
    )
    sweep_eval = evaluate_wildtrack_ground_plane(
        trajectories=sweep_trajectories,
        annotations_dir=ANNOTATIONS_DIR,
        calibrations_dir=CALIBRATIONS_DIR,
        conf_threshold=detection_conf_threshold,
        match_threshold_cm=MATCH_THRESHOLD_CM,
        nms_radius_cm=NMS_RADIUS_CM,
        frame_range=(pred_frame_min, pred_frame_max),
    )
    record = {
        "tracking_params": tracking_params,
        "moda": float(sweep_eval.mota),
        "modp_cm": float(sweep_eval.details.get("modp_cm", 0.0)),
        "idf1": float(sweep_eval.idf1),
        "precision": float(sweep_eval.details.get("precision", 0.0)),
        "recall": float(sweep_eval.details.get("recall", 0.0)),
        "id_switches": int(sweep_eval.id_switches),
        "num_tracks": len(sweep_tracks),
        "num_trajectories": len(sweep_trajectories),
        "num_interpolated_detections": int(interpolation_added),
    }
    return sweep_tracks, sweep_tracklets_by_camera, sweep_trajectories, record


TRACKING_SWEEP_NAIVE_BASELINE = None
TRACKING_SWEEP_KALMAN_BASELINE = None
TRACKING_SWEEP_RESULTS = []
BEST_TRACKING_SWEEP_RESULT = None

naive_params = {
    "max_match_distance_cm": MAX_MATCH_DISTANCE_CM,
    "max_missed_frames": MAX_MISSED_FRAMES,
    "min_track_length": MIN_TRACK_LENGTH,
    "interpolation_max_gap": TRACK_INTERPOLATION_MAX_GAP,
    "detection_conf_threshold": DETECTION_CONF_THRESHOLD,
}
_, _, _, TRACKING_SWEEP_NAIVE_BASELINE = evaluate_tracking_config(False, naive_params)

if USE_KALMAN_TRACKER:
    working_params = {
        "max_age": KALMAN_MAX_AGE,
        "distance_gate": KALMAN_DISTANCE_GATE,
        "min_hits": KALMAN_MIN_HITS,
        "q_std": KALMAN_Q_STD,
        "r_std": KALMAN_R_STD,
        "interpolation_max_gap": TRACK_INTERPOLATION_MAX_GAP,
        "detection_conf_threshold": DETECTION_CONF_THRESHOLD,
    }
    _, _, _, TRACKING_SWEEP_KALMAN_BASELINE = evaluate_tracking_config(True, working_params)

    tracking_sweep_parameters = list(KALMAN_TRACKING_SWEEP.items()) + [("detection_conf_threshold", DETECTION_CONF_SWEEP)]
    for param_name, values in tracking_sweep_parameters:
        stage_best = None
        for value in values:
            candidate_params = dict(working_params)
            candidate_params[param_name] = value
            candidate_tracks, candidate_tracklets, candidate_trajectories, candidate_record = evaluate_tracking_config(
                True,
                candidate_params,
            )
            candidate_record["sweep_parameter"] = param_name
            candidate_record["sweep_value"] = value
            TRACKING_SWEEP_RESULTS.append(candidate_record)
            if stage_best is None or tracking_result_key(candidate_record) > tracking_result_key(stage_best["record"]):
                stage_best = {
                    "tracks": candidate_tracks,
                    "tracklets": candidate_tracklets,
                    "trajectories": candidate_trajectories,
                    "record": candidate_record,
                }

        working_params[param_name] = stage_best["record"]["tracking_params"][param_name]
        print(
            f"[12b] best {param_name}: {working_params[param_name]} -> "
            f"IDF1={stage_best['record']['idf1']:.4f}, MODA={stage_best['record']['moda']:.4f}, "
            f"IDSW={stage_best['record']['id_switches']}"
        )

    tracks, tracklets_by_camera, trajectories, BEST_TRACKING_SWEEP_RESULT = evaluate_tracking_config(True, working_params)
    ACTIVE_TRACKING_PARAMS = BEST_TRACKING_SWEEP_RESULT["tracking_params"]
    _save_ground_plane_tracks(tracks, OUTPUT_DIR / "ground_plane_tracks.json")
    _save_ground_plane_csv(tracks, OUTPUT_DIR / "ground_plane_tracks.csv")
    save_tracklets_by_camera(tracklets_by_camera, OUTPUT_DIR / "tracklets")
    save_global_trajectories(trajectories, OUTPUT_DIR / "global_trajectories.json")
else:
    BEST_TRACKING_SWEEP_RESULT = TRACKING_SWEEP_NAIVE_BASELINE

tracking_sweep_payload = {
    "naive_baseline": TRACKING_SWEEP_NAIVE_BASELINE,
    "kalman_baseline": TRACKING_SWEEP_KALMAN_BASELINE,
    "results": TRACKING_SWEEP_RESULTS,
    "best_result": BEST_TRACKING_SWEEP_RESULT,
}
with (OUTPUT_DIR / "tracking_sweep_results.json").open("w", encoding="utf-8") as handle:
    json.dump(tracking_sweep_payload, handle, indent=2)
with (OUTPUT_DIR / "tracking_sweep_best.json").open("w", encoding="utf-8") as handle:
    json.dump(BEST_TRACKING_SWEEP_RESULT, handle, indent=2)

print(
    f"[12b] naive baseline IDF1: {TRACKING_SWEEP_NAIVE_BASELINE['idf1']:.4f}, "
    f"IDSW={TRACKING_SWEEP_NAIVE_BASELINE['id_switches']}"
)
if TRACKING_SWEEP_KALMAN_BASELINE is not None:
    print(
        f"[12b] kalman baseline IDF1: {TRACKING_SWEEP_KALMAN_BASELINE['idf1']:.4f}, "
        f"IDSW={TRACKING_SWEEP_KALMAN_BASELINE['id_switches']}"
    )
print(f"[12b] selected tracking params: {json.dumps(ACTIVE_TRACKING_PARAMS, indent=2)}")

In [ ]:
import json

import networkx as nx

from src.core.data_models import GlobalTrajectory
from src.stage5_evaluation.ground_plane_eval import evaluate_wildtrack_ground_plane


def trajectory_frame_ids(trajectory):
    return {
        frame.frame_id
        for tracklet in trajectory.tracklets
        for frame in tracklet.frames
    }


def build_merged_trajectories(base_trajectories, candidates, threshold):
    trajectory_by_id = {trajectory.global_id: trajectory for trajectory in base_trajectories}
    graph = nx.Graph()
    graph.add_nodes_from(trajectory_by_id)
    active_candidates = []

    for candidate in candidates:
        score = float(candidate["cosine_similarity"])
        if score < threshold:
            continue
        global_id_a = int(candidate["global_id_a"])
        global_id_b = int(candidate["global_id_b"])
        if global_id_a not in trajectory_by_id or global_id_b not in trajectory_by_id:
            continue
        graph.add_edge(global_id_a, global_id_b, weight=score)
        active_candidates.append(candidate)

    merged_trajectories = []
    applied_merges = []
    components = sorted(
        (sorted(component) for component in nx.connected_components(graph)),
        key=lambda members: members[0],
    )

    for members in components:
        valid_members = [member for member in members if member in trajectory_by_id]
        if not valid_members:
            continue
        member_set = set(valid_members)
        component_candidates = [
            candidate
            for candidate in active_candidates
            if int(candidate["global_id_a"]) in member_set and int(candidate["global_id_b"]) in member_set
        ]
        component_tracklets = [
            tracklet
            for member in valid_members
            for tracklet in trajectory_by_id[member].tracklets
        ]
        component_tracklets = sorted(
            component_tracklets,
            key=lambda tracklet: (tracklet.start_time, tracklet.camera_id, tracklet.track_id),
        )
        edge_scores = [float(candidate["cosine_similarity"]) for candidate in component_candidates]
        if edge_scores:
            confidence = float(np.mean(edge_scores))
        else:
            confidence = float(np.mean([trajectory_by_id[member].confidence for member in valid_members]))
        evidence = [
            {
                "tracklet_a": f"(global, {int(candidate['global_id_a'])})",
                "tracklet_b": f"(global, {int(candidate['global_id_b'])})",
                "similarity": float(candidate["cosine_similarity"]),
                "merge_stage": "reid_temporal_graph",
            }
            for candidate in component_candidates
        ]
        merged_trajectory = GlobalTrajectory(
            global_id=valid_members[0],
            tracklets=component_tracklets,
            confidence=confidence,
            evidence=evidence,
            timeline=[
                {
                    "camera_id": tracklet.camera_id,
                    "start": tracklet.start_time,
                    "end": tracklet.end_time,
                }
                for tracklet in component_tracklets
            ],
        )
        merged_trajectories.append(merged_trajectory)

        if len(valid_members) > 1:
            applied_merges.append(
                {
                    "merged_global_id": valid_members[0],
                    "members": valid_members,
                    "num_tracklets": len(component_tracklets),
                    "confidence": round(confidence, 6),
                    "edges": component_candidates,
                }
            )

    return merged_trajectories, applied_merges, active_candidates


trajectory_by_id = {trajectory.global_id: trajectory for trajectory in trajectories}
trajectory_frames = {
    global_id: trajectory_frame_ids(trajectory)
    for global_id, trajectory in trajectory_by_id.items()
}

REID_TEMPORAL_VALID_CANDIDATES = []
rejected_temporal_overlap = []
for candidate in merge_candidates:
    global_id_a = int(candidate["global_id_a"])
    global_id_b = int(candidate["global_id_b"])
    overlap_frames = sorted(trajectory_frames.get(global_id_a, set()) & trajectory_frames.get(global_id_b, set()))
    if overlap_frames:
        rejected_temporal_overlap.append(
            {
                "global_id_a": global_id_a,
                "global_id_b": global_id_b,
                "cosine_similarity": float(candidate["cosine_similarity"]),
                "overlap_frame_count": len(overlap_frames),
                "overlap_frames_preview": overlap_frames[:10],
            }
        )
        continue
    REID_TEMPORAL_VALID_CANDIDATES.append(candidate)

pred_frame_min = min(frame_ids)
pred_frame_max = max(frame_ids)
active_detection_conf_threshold = (
    float(ACTIVE_TRACKING_PARAMS.get("detection_conf_threshold", DETECTION_CONF_THRESHOLD))
    if "ACTIVE_TRACKING_PARAMS" in globals()
    else float(DETECTION_CONF_THRESHOLD)
)

baseline_eval = evaluate_wildtrack_ground_plane(
    trajectories=trajectories,
    annotations_dir=ANNOTATIONS_DIR,
    calibrations_dir=CALIBRATIONS_DIR,
    conf_threshold=active_detection_conf_threshold,
    match_threshold_cm=MATCH_THRESHOLD_CM,
    nms_radius_cm=NMS_RADIUS_CM,
    frame_range=(pred_frame_min, pred_frame_max),
)
REID_MERGE_BASELINE = {
    "threshold": None,
    "conf_threshold": active_detection_conf_threshold,
    "moda": float(baseline_eval.mota),
    "idf1": float(baseline_eval.idf1),
    "precision": float(baseline_eval.details.get("precision", 0.0)),
    "recall": float(baseline_eval.details.get("recall", 0.0)),
    "id_switches": int(baseline_eval.id_switches),
    "num_trajectories": len(trajectories),
}

REID_MERGE_SWEEP_RESULTS = []
merge_results_by_threshold = {}
best_merge_result = None

for threshold in REID_MERGE_THRESHOLD_SWEEP:
    merged_trajectories, applied_merges, active_candidates = build_merged_trajectories(
        base_trajectories=trajectories,
        candidates=REID_TEMPORAL_VALID_CANDIDATES,
        threshold=threshold,
    )
    sweep_eval = evaluate_wildtrack_ground_plane(
        trajectories=merged_trajectories,
        annotations_dir=ANNOTATIONS_DIR,
        calibrations_dir=CALIBRATIONS_DIR,
        conf_threshold=active_detection_conf_threshold,
        match_threshold_cm=MATCH_THRESHOLD_CM,
        nms_radius_cm=NMS_RADIUS_CM,
        frame_range=(pred_frame_min, pred_frame_max),
    )
    merge_count = int(sum(len(item["members"]) - 1 for item in applied_merges))
    record = {
        "threshold": float(threshold),
        "conf_threshold": active_detection_conf_threshold,
        "moda": float(sweep_eval.mota),
        "idf1": float(sweep_eval.idf1),
        "precision": float(sweep_eval.details.get("precision", 0.0)),
        "recall": float(sweep_eval.details.get("recall", 0.0)),
        "id_switches": int(sweep_eval.id_switches),
        "num_trajectories": len(merged_trajectories),
        "num_merge_edges": len(active_candidates),
        "num_merges_applied": merge_count,
    }
    REID_MERGE_SWEEP_RESULTS.append(record)
    merge_results_by_threshold[threshold] = {
        "trajectories": merged_trajectories,
        "applied_merges": applied_merges,
        "active_candidates": active_candidates,
        "record": record,
    }
    if best_merge_result is None or (record["idf1"], record["threshold"]) > (
        best_merge_result["idf1"],
        best_merge_result["threshold"],
    ):
        best_merge_result = record

assert best_merge_result is not None, "ReID merge sweep produced no results"
BEST_REID_MERGE_THRESHOLD = float(best_merge_result["threshold"])
MERGED_TRAJECTORIES = merge_results_by_threshold[BEST_REID_MERGE_THRESHOLD]["trajectories"]
APPLIED_MERGES = merge_results_by_threshold[BEST_REID_MERGE_THRESHOLD]["applied_merges"]
BEST_REID_MERGE_RESULT = merge_results_by_threshold[BEST_REID_MERGE_THRESHOLD]["record"]

save_global_trajectories(MERGED_TRAJECTORIES, OUTPUT_DIR / "global_trajectories.json")

with (OUTPUT_DIR / "reid_merge_sweep_results.json").open("w", encoding="utf-8") as handle:
    json.dump(
        {
            "baseline": REID_MERGE_BASELINE,
            "temporal_valid_candidates": len(REID_TEMPORAL_VALID_CANDIDATES),
            "temporal_rejected_candidates": len(rejected_temporal_overlap),
            "results": REID_MERGE_SWEEP_RESULTS,
        },
        handle,
        indent=2,
    )
with (OUTPUT_DIR / "reid_merge_sweep_best.json").open("w", encoding="utf-8") as handle:
    json.dump(best_merge_result, handle, indent=2)
with (OUTPUT_DIR / "reid_merges_applied.json").open("w", encoding="utf-8") as handle:
    json.dump(
        {
            "best_threshold": BEST_REID_MERGE_THRESHOLD,
            "baseline": REID_MERGE_BASELINE,
            "best_result": BEST_REID_MERGE_RESULT,
            "temporal_valid_candidates": len(REID_TEMPORAL_VALID_CANDIDATES),
            "temporal_rejected_candidates": len(rejected_temporal_overlap),
            "applied_merges": APPLIED_MERGES,
        },
        handle,
        indent=2,
    )

print(f"[12b] temporal valid merge candidates: {len(REID_TEMPORAL_VALID_CANDIDATES)} / {len(merge_candidates)}")
print(f"[12b] temporal overlap rejections: {len(rejected_temporal_overlap)}")
print(f"[12b] detection confidence threshold: {active_detection_conf_threshold:.2f}")
print(f"[12b] baseline IDF1: {REID_MERGE_BASELINE['idf1']:.4f}")
for record in REID_MERGE_SWEEP_RESULTS:
    print(
        "[12b] threshold="
        f"{record['threshold']:.2f} -> IDF1={record['idf1']:.4f}, "
        f"MODA={record['moda']:.4f}, merges={record['num_merges_applied']}"
    )
print(f"[12b] selected merge threshold: {BEST_REID_MERGE_THRESHOLD:.2f}")
print(f"[12b] merged trajectories: {len(trajectories)} -> {len(MERGED_TRAJECTORIES)}")
for merge in APPLIED_MERGES[:10]:
    print(merge)

In [ ]:
import json

from src.stage5_evaluation.ground_plane_eval import evaluate_wildtrack_ground_plane

pred_frame_min = min(frame_ids)
pred_frame_max = max(frame_ids)
eval_trajectories = MERGED_TRAJECTORIES if "MERGED_TRAJECTORIES" in globals() else trajectories
active_detection_conf_threshold = (
    float(ACTIVE_TRACKING_PARAMS.get("detection_conf_threshold", DETECTION_CONF_THRESHOLD))
    if "ACTIVE_TRACKING_PARAMS" in globals()
    else float(DETECTION_CONF_THRESHOLD)
)

eval_result = evaluate_wildtrack_ground_plane(
    trajectories=eval_trajectories,
    annotations_dir=ANNOTATIONS_DIR,
    calibrations_dir=CALIBRATIONS_DIR,
    conf_threshold=active_detection_conf_threshold,
    match_threshold_cm=MATCH_THRESHOLD_CM,
    nms_radius_cm=NMS_RADIUS_CM,
    frame_range=(pred_frame_min, pred_frame_max),
)

precision = float(eval_result.details.get("precision", 0.0))
recall = float(eval_result.details.get("recall", 0.0))
modp_cm = float(eval_result.details.get("modp_cm", 0.0))
num_merges_applied = int(sum(len(item["members"]) - 1 for item in APPLIED_MERGES)) if "APPLIED_MERGES" in globals() else 0
best_merge_threshold = BEST_REID_MERGE_THRESHOLD if "BEST_REID_MERGE_THRESHOLD" in globals() else None

print("=" * 60)
print("WILDTRACK Ground-Plane Evaluation")
print("=" * 60)
print(f"Trajectories: {len(eval_trajectories)}")
print(f"MODA:        {eval_result.mota:.4f}")
print(f"MODP (cm):   {modp_cm:.4f}")
print(f"IDF1:        {eval_result.idf1:.4f}")
print(f"Precision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"ID Switches: {eval_result.id_switches}")
print(f"Detection conf threshold: {active_detection_conf_threshold:.2f}")
print(f"Tracking params: {json.dumps(ACTIVE_TRACKING_PARAMS, indent=2)}")
if best_merge_threshold is not None:
    print(f"Best ReID merge threshold: {best_merge_threshold:.2f}")
    print(f"Applied merges: {num_merges_applied}")

evaluation_summary = {
    "detection_conf_threshold": active_detection_conf_threshold,
    "moda": float(eval_result.mota),
    "modp_cm": modp_cm,
    "idf1": float(eval_result.idf1),
    "precision": precision,
    "recall": recall,
    "id_switches": int(eval_result.id_switches),
    "num_trajectories": len(eval_trajectories),
    "num_tracklets": sum(len(items) for items in tracklets_by_camera.values()),
    "num_reid_features": len(REID_FEATURES) if REID_FEATURES else 0,
    "num_reid_merge_candidates": len(merge_candidates),
    "num_temporal_valid_merge_candidates": len(REID_TEMPORAL_VALID_CANDIDATES) if "REID_TEMPORAL_VALID_CANDIDATES" in globals() else 0,
    "num_merges_applied": num_merges_applied,
    "best_reid_merge_threshold": best_merge_threshold,
    "reid_merge_baseline": REID_MERGE_BASELINE if "REID_MERGE_BASELINE" in globals() else None,
    "reid_merge_sweep_results": REID_MERGE_SWEEP_RESULTS if "REID_MERGE_SWEEP_RESULTS" in globals() else [],
    "tracking_params": ACTIVE_TRACKING_PARAMS,
    "tracking_sweep_best": BEST_TRACKING_SWEEP_RESULT if "BEST_TRACKING_SWEEP_RESULT" in globals() else None,
    "tracking_sweep_baseline_naive": TRACKING_SWEEP_NAIVE_BASELINE if "TRACKING_SWEEP_NAIVE_BASELINE" in globals() else None,
    "tracking_sweep_baseline_kalman": TRACKING_SWEEP_KALMAN_BASELINE if "TRACKING_SWEEP_KALMAN_BASELINE" in globals() else None,
}

with (OUTPUT_DIR / "evaluation_summary.json").open("w", encoding="utf-8") as handle:
    json.dump(evaluation_summary, handle, indent=2)

In [ ]:
import json

tracking_summary = {
    "naive_baseline": TRACKING_SWEEP_NAIVE_BASELINE if "TRACKING_SWEEP_NAIVE_BASELINE" in globals() else None,
    "kalman_baseline": TRACKING_SWEEP_KALMAN_BASELINE if "TRACKING_SWEEP_KALMAN_BASELINE" in globals() else None,
    "best_result": BEST_TRACKING_SWEEP_RESULT if "BEST_TRACKING_SWEEP_RESULT" in globals() else None,
    "num_sweep_runs": len(TRACKING_SWEEP_RESULTS) if "TRACKING_SWEEP_RESULTS" in globals() else 0,
}
print(json.dumps(tracking_summary, indent=2))

In [ ]:
import json
import shutil

key_artifacts = [
    OUTPUT_DIR / "ground_plane_tracks.json",
    OUTPUT_DIR / "ground_plane_tracks.csv",
    OUTPUT_DIR / "global_trajectories.json",
    OUTPUT_DIR / "evaluation_summary.json",
    OUTPUT_DIR / "tracking_sweep_best.json",
    OUTPUT_DIR / "reid_merge_candidates.json",
    OUTPUT_DIR / "reid_merges_applied.json",
    OUTPUT_DIR / "reid_merge_sweep_results.json",
    OUTPUT_DIR / "reid_merge_sweep_best.json",
    OUTPUT_DIR / "reid_features.npz",
]

print("=" * 60)
print("12b Output Artifacts")
print("=" * 60)
output_listing = []
for path in sorted(OUTPUT_DIR.rglob("*")):
    if not path.is_file():
        continue
    size_mb = path.stat().st_size / (1024 * 1024)
    rel_path = str(path.relative_to(OUTPUT_DIR))
    output_listing.append({"path": rel_path, "size_mb": round(size_mb, 4)})
    print(f"{rel_path}: {size_mb:.2f} MB")

for artifact_path in key_artifacts:
    if artifact_path.is_file():
        shutil.copy2(artifact_path, Path("/kaggle/working") / artifact_path.name)

summary = {
    "output_dir": str(OUTPUT_DIR),
    "artifact_count": len(output_listing),
    "artifacts": output_listing,
    "copied_to_working": [path.name for path in key_artifacts if path.is_file()],
}
with (OUTPUT_DIR / "artifact_manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2)
print(json.dumps(summary, indent=2))